# Agent Execution Sequence

This notebook illustrates the internal sequence of operations defined in `0_create_agent.py` when the user invokes the agent with multiple math queries.

```mermaid
sequenceDiagram
    actor User
    participant Graph as Agent Graph
    participant LLM as Gemini 2.5 Flash
    participant Tool as Calculator Tool
    participant State as CalcState

    User->>Graph: invoke({"messages": ["what is 2/1 and 4+7 and 6/3"]})
    
    Note over Graph,LLM: Step 1: Initial LLM Inference
    Graph->>LLM: predict(messages)
    LLM-->>Graph: tool_call: calculator_wstate(DIV, 2, 1)
    
    Note over Graph,State: Step 2: First Tool Execution
    Graph->>Tool: execute(DIV, 2, 1)
    Tool->>State: Command(update={'ops': ['(DIV, 2, 1) = 2.0']})
    Tool-->>Graph: ToolMessage("2.0")
    
    Note over Graph,LLM: Step 3: Second LLM Inference
    Graph->>LLM: predict(messages + ToolMessage(2.0))
    LLM-->>Graph: tool_call: calculator_wstate(ADD, 4, 7)
    
    Note over Graph,State: Step 4: Second Tool Execution
    Graph->>Tool: execute(ADD, 4, 7)
    Tool->>State: Command(update={'ops': ['(ADD, 4, 7) = 11.0']})
    Tool-->>Graph: ToolMessage("11.0")
    
    Note over Graph,LLM: Step 5: Third LLM Inference
    Graph->>LLM: predict(messages + ToolMessage(11.0))
    LLM-->>Graph: tool_call: calculator_wstate(DIV, 6, 3)
    
    Note over Graph,State: Step 6: Third Tool Execution
    Graph->>Tool: execute(DIV, 6, 3)
    Tool->>State: Command(update={'ops': ['(DIV, 6, 3) = 2.0']})
    Tool-->>Graph: ToolMessage("2.0")
    
    Note over Graph,LLM: Step 7: Final LLM Synthesis
    Graph->>LLM: predict(messages + ToolMessage(2.0))
    LLM-->>Graph: AIMessage("The results are 2.0, 11.0, and 2.0.")
    
    Graph-->>User: return final state (messages)
```
